# Module 1: Dataset Exploration & Integrity Audit
**Project Title**: Leaf Quality Classification and Plucking Standard Monitoring System Using Handheld Camera  
**Course**: CSE471 Academic Project  
**Author / Team**: Senior AI/ML Engineer & CSE471 Project Team  

---

### Notebook Overview
In this notebook, we perform the initial Exploratory Data Analysis (EDA) and dataset auditing for Module 1. We inspect:
1. **PlantDoc Dataset**: In-field, handheld camera images with complex backgrounds, occlusion, and real-world lighting variations.
2. **New Plant Diseases Dataset**: Controlled-environment benchmark images showing various plant leaves in healthy and diseased states.

### Objectives:
- Import computer vision and data analysis libraries.
- Define canonical paths for raw datasets.
- Scan and catalog classes, image counts, and file formats.
- Visualize class distributions and sample leaf images.
- Analyze image resolutions, aspect ratios, and detect corrupted files.
- Document critical findings regarding dataset mappings and identify missing requirements (e.g., the *Not Ready* / immature leaf category).


In [ ]:
# Cell 1: Library Imports
import os
import sys
from pathlib import Path
from typing import Dict, List, Tuple

# Matplotlib configuration to avoid environment permission warnings
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / 'reports' / '.mplconfig'))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Check environment versions
print(f'NumPy Version      : {np.__version__}')
print(f'Pandas Version     : {pd.__version__}')
print(f'OpenCV Version     : {cv2.__version__}')
print(f'Pillow Version     : {Image.__version__}')
print(f'Matplotlib Version : {plt.matplotlib.__version__}')


## 2. Dataset Paths and Configuration
We define the canonical directories where raw datasets are located. The user downloads the datasets from Kaggle and places them inside `datasets/raw/plantdoc` and `datasets/raw/new_plant_diseases`.


In [ ]:
# Cell 2: Define Paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

DATASETS_RAW = PROJECT_ROOT / 'datasets' / 'raw'
PLANTDOC_DIR = DATASETS_RAW / 'plantdoc'
NEW_PLANT_DISEASES_DIR = DATASETS_RAW / 'new_plant_diseases'
REPORTS_DIR = PROJECT_ROOT / 'reports'

print(f'Project Root       : {PROJECT_ROOT}')
print(f'PlantDoc Path      : {PLANTDOC_DIR} (Exists: {PLANTDOC_DIR.exists()})')
print(f'New Diseases Path  : {NEW_PLANT_DISEASES_DIR} (Exists: {NEW_PLANT_DISEASES_DIR.exists()})')
print(f'Reports Path       : {REPORTS_DIR}')


## 3. Dataset Scanning & Integrity Verification Function
We define helper functions to:
1. Test image readability and detect corruptions using both **Pillow** (`img.verify()`) and **OpenCV** (`cv2.imread()`).
2. Catalog class folders, supported image formats, and image dimensions.


In [ ]:
# Cell 3: Scanning and Integrity Verification Utilities
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff', '.tif'}

def check_image_health(img_path: Path):
    """Verifies image integrity and extracts dimensions."""
    try:
        with Image.open(img_path) as img:
            img.verify()
    except Exception:
        return True, (0, 0, 0)
    
    try:
        cv_img = cv2.imread(str(img_path))
        if cv_img is None:
            return True, (0, 0, 0)
        h, w = cv_img.shape[:2]
        c = cv_img.shape[2] if len(cv_img.shape) > 2 else 1
        return False, (h, w, c)
    except Exception:
        return True, (0, 0, 0)

def scan_directory(dataset_name: str, base_path: Path):
    """Scans dataset directory and returns structured records."""
    records = []
    if not base_path.exists():
        return records

    subdirs = [p for p in base_path.glob('**/*') if p.is_dir()]
    all_folders = [base_path] + subdirs

    for folder in all_folders:
        files = [f for f in folder.iterdir() if f.is_file() and f.name != '.gitkeep']
        image_files = [f for f in files if f.suffix.lower() in VALID_EXTENSIONS]

        rel_path = folder.relative_to(base_path)
        class_name = 'root' if str(rel_path) == '.' else str(rel_path).replace('\\', '/')

        # Skip intermediate non-leaf directories
        has_subdirs = any(p.is_dir() for p in folder.iterdir())
        if has_subdirs and len(image_files) == 0:
            continue

        if not image_files:
            continue

        corrupted = 0
        formats = set()
        heights, widths = [], []
        sample_dim = 'N/A'

        for img_p in image_files:
            formats.add(img_p.suffix.lower())
            is_corrupt, (h, w, c) = check_image_health(img_p)
            if is_corrupt:
                corrupted += 1
            else:
                heights.append(h)
                widths.append(w)
                if sample_dim == 'N/A':
                    sample_dim = f'{w}x{h}x{c}'

        dim_stats = 'N/A'
        if heights and widths:
            dim_stats = f'Avg: {int(np.mean(widths))}x{int(np.mean(heights))}, Min: {min(widths)}x{min(heights)}, Max: {max(widths)}x{max(heights)}'

        records.append({
            'Dataset Name': dataset_name,
            'Original Class Name': class_name,
            'Number of Images': len(image_files),
            'Image Formats': ', '.join(sorted(formats)),
            'Sample Dimensions': sample_dim,
            'Dimension Statistics': dim_stats,
            'Corrupted Images': corrupted,
            'Sample Image Path': str(image_files[0]) if image_files else None
        })
    return records


## 4. Loading Dataset Information & Summary Table
Let's run the scan across both datasets and display the results in a structured Pandas DataFrame.


In [ ]:
# Cell 4: Execute Dataset Scan
plantdoc_records = scan_directory('PlantDoc', PLANTDOC_DIR)
new_diseases_records = scan_directory('New Plant Diseases', NEW_PLANT_DISEASES_DIR)

all_records = plantdoc_records + new_diseases_records

if all_records:
    df_inventory = pd.DataFrame(all_records)
    print(f'Total Categories Cataloged: {len(df_inventory)}')
    print(f'Total Images Found        : {df_inventory["Number of Images"].sum():,}')
    print(f'Total Corrupted Images    : {df_inventory["Corrupted Images"].sum():,}')
    print(df_inventory[['Dataset Name', 'Original Class Name', 'Number of Images', 'Image Formats', 'Sample Dimensions', 'Corrupted Images']].head(15))
else:
    print('[!] Datasets have not been placed in the raw directories yet.')
    print('    PlantDoc Location        :', PLANTDOC_DIR)
    print('    New Plant Diseases Path  :', NEW_PLANT_DISEASES_DIR)
    print('\n>> Please download the datasets from Kaggle and extract them to the above paths.')
    df_inventory = pd.DataFrame(columns=['Dataset Name', 'Original Class Name', 'Number of Images', 'Image Formats', 'Sample Dimensions', 'Corrupted Images'])


## 5. Class Distribution Visualization
Visualizing the distribution of images per category helps identify class imbalances, which informs preprocessing and augmentation strategies in Module 2.


In [ ]:
# Cell 5: Class Distribution Chart
if not df_inventory.empty and df_inventory['Number of Images'].sum() > 0:
    top_classes = df_inventory.sort_values(by='Number of Images', ascending=False).head(25)
    plt.figure(figsize=(12, 7))
    plt.barh(top_classes['Original Class Name'], top_classes['Number of Images'], color='#2e7d32')
    plt.xlabel('Image Count', fontsize=11)
    plt.ylabel('Class Name', fontsize=11)
    plt.title('Top 25 Leaf Disease / Category Distribution', fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('No images available to display distribution chart. Place raw datasets to view.')


## 6. Displaying Sample Images from Available Categories
Visual inspection of sample images allows us to examine background clutter, lighting variations, leaf orientation, and visual symptoms.


In [ ]:
# Cell 6: Visual Inspection of Sample Images
if not df_inventory.empty and 'Sample Image Path' in df_inventory.columns:
    sample_rows = df_inventory[df_inventory['Sample Image Path'].notnull()].head(6)
    if not sample_rows.empty:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        for idx, (_, row) in enumerate(sample_rows.iterrows()):
            img_bgr = cv2.imread(row['Sample Image Path'])
            if img_bgr is not None:
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                axes[idx].imshow(img_rgb)
                axes[idx].set_title(f"{row['Dataset Name']}\n{row['Original Class Name'][:30]}", fontsize=10)
                axes[idx].axis('off')
            else:
                axes[idx].text(0.5, 0.5, 'Unreadable Image', ha='center')
                axes[idx].axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print('No sample images found to display.')
else:
    print('Raw datasets not yet populated. Sample preview will appear after datasets are placed.')


## 7. Image Dimensions & Aspect Ratio Analysis
Understanding image resolutions is critical for selecting optimal input dimensions for our planned **MobileNetV2** model (standard input: $224 \times 224 \times 3$).


In [ ]:
# Cell 7: Dimension Inspection
if not df_inventory.empty and 'Dimension Statistics' in df_inventory.columns:
    print('Image Dimension Summary by Class:')
    for _, row in df_inventory.head(10).iterrows():
        print(f"[{row['Dataset Name']}] {row['Original Class Name']}: {row['Dimension Statistics']}")
else:
    print('No dimension statistics available yet. Once images are placed, resolution metrics will be computed.')


## 8. Corrupted Image Audit
Ensuring zero corruptions prevents batch failures, pipeline stalls, and data loading exceptions during training.


In [ ]:
# Cell 8: Check Corrupted Images
if not df_inventory.empty and 'Corrupted Images' in df_inventory.columns:
    corrupted_df = df_inventory[df_inventory['Corrupted Images'] > 0]
    if not corrupted_df.empty:
        print(f'[!] Warning: Found {len(corrupted_df)} folder(s) with corrupted images:')
        print(corrupted_df[['Dataset Name', 'Original Class Name', 'Corrupted Images']])
    else:
        print('[✓] Integrity Check Passed: Zero corrupted images detected across all scanned folders.')
else:
    print('Integrity check awaiting dataset placement.')


## 9. Observations & Critical CSE471 Architectural Insights

### Findings & Category Mapping Analysis:
1. **Preserving Original Class Names**:
   Original dataset classes (e.g. `Tomato___healthy`, `Apple___Apple_scab`, `Corn___Northern_Leaf_Blight`) must remain intact in raw storage.
2. **Quality Category Mapping**:
   Our project target classification has three defined outcomes:
   - **Good Quality**: Mature, healthy leaves meeting plucking standards.
   - **Poor Quality**: Diseased, pest-damaged, scorched, or defective leaves that should not be plucked.
   - **Not Ready**: Immature, young, tender leaves that require further growth before harvesting.
3. **The 'Not Ready' Category Gap**:
   Neither PlantDoc nor New Plant Diseases explicitly annotates physiological leaf maturity (e.g. 'tender bud', 'immature flush', 'not ready for harvest').
   - *Requirement*: The 'Not Ready' class cannot be synthesized or guessed. We must clearly flag this as a missing dataset requirement to be addressed through targeted collection of young/immature leaf images before training in Module 2.
4. **Model Input Sizing**:
   Standardizing input resolutions (e.g., resizing to $224 \times 224$) with aspect-ratio-preserving padding or standard scaling will be essential for the MobileNetV2 architecture.

---
**Module 1 Status**: Completed and ready for manual dataset placement and subsequent Module 2 preprocessing.
